# Stage Four — Intervention Economics

## Position within the project

This is the last of four notebooks describing a project which predicts which customers of a telecommunications provider are likely to leave, and establishes which of them are worth the cost of retaining. The project and the source data are introduced in `01_data_inventory.ipynb`; the four notebooks form one continuous account, read in sequence.

Two components are now complete. Stage Three produced a model estimating the likelihood that each customer will depart. Stage Two produced a calculated value for each customer.

This stage combines them, and in doing so addresses a different question from the one the model answers. The model establishes which customers are likely to depart. This stage establishes which customers justify the cost of retaining them. These prove to be substantially different populations.

That distinction is the reason the project exists. A churn model in isolation produces a list of customers likely to depart, and offers no indication of whether pursuing them is commercially sound. The results below establish that pursuing the majority of them is not.

**Division of responsibility.** The decision rule and the assumptions supporting it are held in `sql/05_economics.sql`. Python is used only to execute that file and retrieve the results.

---

## 1. Combining the two components

The model's scores are joined to the customer records, so that every customer carries both a probability of departure and a calculated value.

This is performed as a database operation rather than assembled within Python, consistent with the division of responsibility maintained throughout the project.

In [1]:
import duckdb
import pandas as pd

con = duckdb.connect("../telco.duckdb")

econ = con.execute("""
    SELECT s.customer_id,
           s.clv,
           s.value_decile,
           s.churn_probability,
           s.churn_value,
           f.contract,
           f.tenure_band,
           f.monthly_charges
    FROM customer_scores s
    JOIN vw_customer_features f ON s.customer_id = f.customer_id
""").df()

print(econ.shape)
econ.head()

(7043, 8)


,customer_id,clv,value_decile,churn_probability,churn_value,contract,tenure_band,monthly_charges
0,3668-QPYBK,193.86,8,0.551139,1,Month-to-month,New,53.85
1,9237-HQITU,254.52,7,0.838677,1,Month-to-month,New,70.70
2,9305-CDSKC,358.74,4,0.893799,1,Month-to-month,New,99.65
3,7892-POOKP,377.28,4,0.808888,1,Month-to-month,Established,104.80
4,0280-XJGEX,373.32,4,0.591416,1,Month-to-month,Loyal,103.70


**Result:** 7,043 customers, each carrying a probability of departure alongside a calculated value.

## 2. Applying the decision rule

The rule consists of a single calculation:

> **Value of contacting a customer  =  probability of departure  ×  probability of successful retention  ×  customer value  −  cost of contact**

Where the result is positive, the customer justifies contact. Where it is negative, contacting the customer destroys value — and this remains the case even where the customer genuinely intended to depart.

Two of the figures within that calculation are absent from the dataset. Both are assumptions, and both are recorded within `sql/05_economics.sql` alongside the calculation which relies upon them.

**Cost of contact: R60 per customer.** This comprises R20 for an automated electronic communication, and R40 for the retention offer itself, calculated as R20 per month sustained over two months. The average customer pays R64.76 per month, so the offer represents approximately a 30 percent reduction sustained for a short period. The figure is derived from what customers actually pay rather than selected to produce a favourable outcome.

**Probability of successful retention: 30 percent.** Of customers who would otherwise have departed, approximately three in ten are retained when presented with an offer. This is the least certain figure within the project, and the first which an operating business would replace with results from its own campaigns.

**The cost is incurred for every customer contacted**, including those who were not in fact departing. The model is correct on approximately one identification in two, and this rule incorporates that limitation directly into the arithmetic rather than treating it as a separate concern.

**A note regarding the cost figure.** The rule was initially constructed on an assumption of R340 per customer, representing an agent-led telephone contact together with a larger offer sustained over six months. Under that assumption, not one of the 7,043 customers justified contact. That outcome is deliberately retained within the SQL file, as it constitutes a finding rather than a failure: at the values these customers carry, an expensive retention programme cannot recover its own cost irrespective of how well the model performs. Section 3 presents the analysis which established this.

In [6]:
con.execute(open("../sql/05_economics.sql").read())

con.execute("""
    SELECT decision,
           COUNT(*) AS customers,
           ROUND(SUM(expected_value_saved),2) AS value_saved,
           ROUND(SUM(intervention_cost),2) AS total_cost,
           ROUND(SUM(net_value),2) AS net
    FROM vw_intervention_economics
    GROUP BY 1
""").df()

,decision,customers,value_saved,total_cost,net
0,Do not intervene,4825,105819.35,289500.0,-183680.65
1,Intervene,2218,172742.37,133080.0,39662.37


**Result:** a meaningful division of the customer base.

| Decision | Customers | Value recovered | Cost | Net |
|---|---|---|---|---|
| **Contact** | 2,218 | R172,742 | R133,080 | **+R39,662** |
| Do not contact | 4,825 | R105,819 | R289,500 | −R183,681 |

The second row carries the substance of the argument. Those 4,825 customers would cost R289,500 to contact and would return R105,819. Pursuing them destroys R183,681.

**The comparison between the two available approaches is as follows.**

Contacting the entire customer base recovers R278,562 against expenditure of R422,580. The programme **incurs a net loss of R144,018**.

Contacting only the 2,218 customers identified by the rule **returns R39,662**, representing a 30 percent return on expenditure.

The difference between those outcomes is approximately R184,000. None of it derives from improved model accuracy — the model is identical in both cases. The entire difference arises from determining which customers should not be contacted.

> **Finding.** A churn model considered in isolation indicates that a retention campaign is commercially justified. The economics establish that it is not, unless targeted. The value lay not in the prediction but in the decision regarding whom to exclude.

## 3. Establishing the ceiling on recoverable value

Before any cost assumption can be relied upon, a direct question requires an answer: what is the maximum recoverable from any single customer?

This analysis is what exposed the original R340 assumption.

In [7]:
con.execute("""
    SELECT ROUND(MIN(expected_value_saved),2)  AS min_saved,
           ROUND(AVG(expected_value_saved),2)  AS avg_saved,
           ROUND(MAX(expected_value_saved),2)  AS max_saved,
           ROUND(QUANTILE_CONT(expected_value_saved, 0.90),2) AS p90,
           ROUND(QUANTILE_CONT(expected_value_saved, 0.99),2) AS p99
    FROM vw_intervention_economics
""").df()

,min_saved,avg_saved,max_saved,p90,p99
0,0.56,39.55,163.09,82.28,109.24


**Result:** the recoverable amounts are low.

| | Amount |
|---|---|
| Minimum recoverable from a single customer | R0.56 |
| Typical customer | R39.55 |
| Highest 10 percent of customers | above R82.28 |
| Highest 1 percent of customers | above R109.24 |
| **Maximum across the customer base** | **R163.09** |

The most valuable customer within the business justifies expenditure of at most R163 to retain. The typical customer justifies R40.

These figures eliminated the R340 programme from consideration. The issue was never one of refining the model or improving the offer — no customer within this business carries sufficient value to justify R340 of retention expenditure. The programme required a lower-cost, automated approach, or no programme at all.

> **Finding.** The scale of the available return should be established before a campaign is designed. Had the R340 programme proceeded on the strength of a well-performing model, it would have incurred a loss on every customer it reached.

## 4. Distributing the decision across value groups

The decision is finally examined across the ten value groups established at Stage Two, ranging from the most valuable customers to the least.

The final column records the outcome within each group if every customer in it were contacted without applying the rule.

In [8]:
con.execute("""
    SELECT value_decile,
           COUNT(*) AS customers,
           SUM(CASE WHEN decision = 'Intervene' THEN 1 ELSE 0 END) AS intervene,
           ROUND(SUM(CASE WHEN decision = 'Intervene' THEN net_value ELSE 0 END),2) AS net_if_targeted,
           ROUND(SUM(net_value),2) AS net_if_all_contacted
    FROM vw_intervention_economics
    GROUP BY 1 ORDER BY 1
""").df()

,value_decile,customers,intervene,net_if_targeted,net_if_all_contacted
0,1,705,83.0,1599.74,-17760.85
1,2,705,349.0,10261.63,-2598.32
2,3,705,138.0,2642.34,-17693.18
3,4,704,589.0,13506.37,11195.62
4,5,704,532.0,8206.15,5215.91
5,6,704,363.0,2974.06,-9179.05
6,7,704,164.0,472.08,-19150.76
7,8,704,0.0,0.00,-28863.49
8,9,704,0.0,0.00,-30433.87
9,10,704,0.0,0.00,-34750.29


**Result:** the output the project was constructed to produce.

| Value group | Customers | Justify contact | Net if targeted | Net if all contacted |
|---|---|---|---|---|
| 1 (most valuable) | 705 | 83 | +R1,600 | −R17,761 |
| 2 | 705 | 349 | +R10,262 | −R2,598 |
| 3 | 705 | 138 | +R2,642 | −R17,693 |
| **4** | 704 | **589** | **+R13,506** | +R11,196 |
| **5** | 704 | **532** | **+R8,206** | +R5,216 |
| 6 | 704 | 363 | +R2,974 | −R9,179 |
| 7 | 704 | 164 | +R472 | −R19,151 |
| 8 | 704 | **0** | R0 | −R28,863 |
| 9 | 704 | **0** | R0 | −R30,434 |
| 10 (least valuable) | 704 | **0** | R0 | −R34,750 |

Three findings arise from this distribution.

**The least valuable 30 percent of customers do not justify contact under any circumstances.** Not one customer within groups 8, 9 and 10 meets the threshold — none of the 2,112 individuals concerned. Contacting that population would destroy R94,048. The rule does not reduce expenditure upon this group; it eliminates it entirely.

**The return is concentrated within groups 4 and 5.** Those 1,408 customers produce R21,712 between them, representing 55 percent of the programme's entire return from 20 percent of the customer base.

**The most valuable customers feature only marginally.** Only 83 of the highest 705 justify contact, as the remainder are not departing. High value combined with low risk leaves correspondingly little to recover. Contacting the entire group would incur a loss of R17,761.

> **Finding.** Ranking customers by likelihood of departure directs attention toward groups 4 to 6. Ranking by customer value directs attention toward groups 1 to 3. Neither list is correct in isolation. The return is located where the two overlap, and only this stage identifies it.